In [ ]:
from pathlib import Path

%pip install --quiet --disable-pip-version-check requests==2.34.2
%pip install --quiet --disable-pip-version-check numpy==2.5.3
%pip install --quiet --disable-pip-version-check pandas==3.0.6
%pip install --quiet --disable-pip-version-check pandas_market_calendars==5.4.0
%pip install --quiet --disable-pip-version-check matplotlib==3.11.2

%pip install --quiet --disable-pip-version-check scikit-learn==1.9.1
%pip install --quiet --disable-pip-version-check threadpoolctl==3.7.0

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import requests
import os
from getpass import getpass
from urllib.parse import quote
from IPython.display import display
from matplotlib.ticker import StrMethodFormatter

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

API_KEY = os.environ.get("EULERPOOL_API_KEY", "").strip()

In [ ]:
SYMBOL = "BTC-USD"
START = "2023-10-01"
END = "2026-10-04"
WINDOW = 30
ANNUAL_DAYS = 365 if SYMBOL.endswith("-USD") else 252
TRAIN_DAYS = ANNUAL_DAYS
SEED = 42
CALENDAR = "24/7" if SYMBOL.endswith("-USD") else "NYSE"

DATA = Path("data")
OUTPUT = Path("results") / "Eulerpool" / SYMBOL
DATA.mkdir(exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

LOWER = "Lower volatility"
HIGHER = "Higher volatility"

METHOD = "Gaussian Mixture regimes"

In [ ]:
def eulerpool_request(path, params=None):
    global API_KEY
    if not API_KEY:
        API_KEY = getpass("Eulerpool API key: ").strip()
    if not API_KEY:
        raise ValueError("An Eulerpool API key is required.")
    try:
        response = requests.get(
            f"https://api.eulerpool.com/api/1/{path}", params=params,
            headers={"Authorization": f"Bearer {API_KEY}", "Accept": "application/json"},
            timeout=(3, 15), allow_redirects=False,
        )
    except requests.RequestException:
        raise ValueError("Eulerpool could not be reached. Try again later.") from None
    if response.status_code in (401, 403):
        raise ValueError("Check your Eulerpool API key and data access.")
    if response.status_code == 429:
        raise ValueError("Eulerpool's request limit was reached. Try again later.")
    if response.status_code != 200:
        raise ValueError(f"Eulerpool could not return this data (status {response.status_code}).")
    try:
        return response.json()
    except ValueError:
        raise ValueError("Eulerpool returned an unreadable response.") from None


def eulerpool_prices(symbol, start, end):
    crypto = symbol.endswith("-USD")
    identifier = quote(symbol.removesuffix("-USD") if crypto else symbol, safe="")
    category = "crypto" if crypto else "equity"
    profile = eulerpool_request(f"{category}/profile/{identifier}")
    if not isinstance(profile, dict) or str(profile.get("currency", "")).upper() != "USD":
        raise ValueError("This experiment needs an asset with prices quoted in USD.")
    start = pd.Timestamp(start).normalize().tz_localize("UTC")
    end = min(pd.Timestamp(end).normalize().tz_localize("UTC"), pd.Timestamp.now(tz="UTC").normalize())
    if start >= end:
        raise ValueError("The end date must be after the start date.")
    if crypto:
        data = eulerpool_request(f"crypto/quotes/{identifier}", {
            "startdate": int(start.timestamp() * 1000), "enddate": int(end.timestamp() * 1000),
        })
        try:
            timestamps = [row["timestamp"] for row in data]
            values = [row["price"] for row in data]
        except (KeyError, TypeError):
            raise ValueError("Eulerpool returned unreadable crypto prices.") from None
        unit = "ms"
    else:
        data = eulerpool_request(f"charting/ohlcv/{identifier}", {
            "resolution": "D", "from": int(start.timestamp()), "to": int(end.timestamp()),
        })
        if not isinstance(data, dict):
            raise ValueError("Eulerpool returned unreadable stock prices.")
        timestamps, values, unit = data.get("t", []), data.get("c", []), "s"
    if not timestamps or len(timestamps) != len(values):
        raise ValueError("Eulerpool did not return a complete price response.")
    try:
        dates = pd.to_datetime(timestamps, unit=unit, utc=True, errors="raise")
        prices = pd.Series(pd.to_numeric(values, errors="raise"), index=dates, dtype=float).sort_index()
    except (ValueError, TypeError, OverflowError):
        raise ValueError("Eulerpool returned invalid prices or dates.") from None
    prices = prices.loc[(prices.index >= start) & (prices.index < end)]
    if prices.empty or prices.index.hasnans or not np.isfinite(prices).all() or prices.le(0).any():
        raise ValueError("Eulerpool returned missing or invalid prices.")
    prices.index = prices.index.tz_localize(None).normalize()
    prices = prices.groupby(level=0).last()
    if not crypto:
        prices = prices.loc[prices.index.dayofweek < 5]
    return prices.rename("Price").rename_axis("Date")

In [ ]:
path = DATA / f"Eulerpool_{SYMBOL}_{START}_{END}.csv"
if not path.exists():
    eulerpool_prices(SYMBOL, START, END).to_csv(path)

prices = pd.read_csv(path, index_col="Date", parse_dates=True)["Price"]
cutoff = min(pd.Timestamp(END), pd.Timestamp.now(tz="UTC").normalize().tz_localize(None))
prices = prices.loc[(prices.index >= pd.Timestamp(START)) & (prices.index < cutoff)]
if not prices.index.is_unique or not prices.index.is_monotonic_increasing:
    raise ValueError("Price dates must be unique and ordered.")
if prices.empty or not np.isfinite(prices).all() or prices.le(0).any():
    raise ValueError("Prices must be positive and complete.")
calendar = mcal.get_calendar(CALENDAR)
expected = calendar.valid_days(pd.Timestamp(START), cutoff - pd.Timedelta(days=1)).tz_localize(None)
if not prices.index.equals(expected):
    missing = expected.difference(prices.index)
    raise ValueError(
        f"Eulerpool has {len(missing)} missing expected price dates, or the calendar does not match. "
        "Use complete daily data before running the experiment. Missing prices are not filled."
    )
if len(prices) <= WINDOW + TRAIN_DAYS:
    raise ValueError("There is not enough price history.")
display(pd.Series({
    "Asset": SYMBOL,
    "Source": "Eulerpool: https://eulerpool.com/developers",
    "Prices": f"{prices.index[0].date()} to {prices.index[-1].date()}",
    "Signal timing": "After the completed UTC day",
    "Price basis": "Last available observation per UTC day; stock adjustments are not verified",
}, name="Data").to_frame())

In [ ]:
returns = prices.pct_change(fill_method=None)
volatility = returns.rolling(WINDOW).std() * np.sqrt(ANNUAL_DAYS) * 100
features = volatility.dropna().to_frame("Volatility %")

In [ ]:
rows = []
with threadpool_limits(limits=1):
    for i in range(TRAIN_DAYS, len(features)):
        training = features.iloc[i - TRAIN_DAYS:i]
        current = features.iloc[[i]]
        scaler = StandardScaler().fit(training)
        model = GaussianMixture(n_components=2, n_init=5, max_iter=500, reg_covar=0.0001, random_state=SEED)
        model.fit(scaler.transform(training))
        if not model.converged_:
            raise ValueError("The regime model did not converge.")
        centers = scaler.inverse_transform(model.means_)[:, 0]
        lower_state, higher_state = np.argsort(centers)
        labels = {lower_state: LOWER, higher_state: HIGHER}
        state = model.predict(scaler.transform(current))[0]
        rows.append({
            "Date": current.index[0],
            "Volatility %": current.iloc[0]["Volatility %"],
            "Regime": labels[state],
            "Training through": training.index[-1],
        })

regimes = pd.DataFrame(rows).set_index("Date")

In [ ]:
regimes.insert(0, "Price", prices.reindex(regimes.index))
previous = regimes["Regime"].shift(1)
regimes["Changed"] = previous.notna() & regimes["Regime"].ne(previous)
changes = regimes.loc[regimes["Changed"]].copy()
changes.insert(1, "Previous regime", previous.loc[changes.index])

display(regimes.tail(10).round(2))

In [ ]:
display(changes.tail(10).round(2))

In [ ]:
colors = {LOWER: "#28745e", HIGHER: "#bb6847"}
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, layout="constrained")
axes[0].plot(regimes.index, regimes["Price"], color="#a5aba8", linewidth=0.8)
axes[1].plot(regimes.index, regimes["Volatility %"], color="#a5aba8", linewidth=0.8)
for label, color in colors.items():
    selected = regimes["Regime"].eq(label)
    axes[0].scatter(regimes.index[selected], regimes.loc[selected, "Price"], s=8, color=color, label=label)
    axes[1].scatter(regimes.index[selected], regimes.loc[selected, "Volatility %"], s=8, color=color)
axes[0].set_ylabel("Daily price (USD)")
axes[0].yaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
axes[0].legend(frameon=False, loc="upper left")
axes[1].set_ylabel("Annualized volatility (%)")
axes[1].set_ylim(bottom=0)
axes[1].set_xlabel("Date")
if "Threshold %" in regimes:
    axes[1].plot(regimes.index, regimes["Threshold %"], color="#3d638d", linestyle="--", linewidth=1.2, label="Previous year's median")
    axes[1].legend(frameon=False, loc="upper right")
for axis in axes:
    axis.grid(axis="y", alpha=0.2)
    axis.margins(x=0.01)
fig.suptitle(f"{SYMBOL} | {METHOD}", fontsize=13)
axes[0].set_title(f"{WINDOW}-day volatility; {TRAIN_DAYS} earlier observations for each signal", fontsize=10, color="#64716b")
fig.savefig(OUTPUT / "regime_changes.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
regimes.to_csv(OUTPUT / "regimes.csv", date_format="%Y-%m-%d")
changes.to_csv(OUTPUT / "regime_changes.csv", date_format="%Y-%m-%d")

This notebook now reads Eulerpool prices and needs to be rerun before we can report new regime changes. Missing daily prices stop the experiment before the model is fitted. With a complete history, we will compare the Gaussian Mixture labels with the price-rule labels over the same period. That comparison can help us decide what to test in the portfolio next; it will not by itself show that the model improves returns. The earlier Yahoo Finance results remain separate.